# 1D Breakout Model Extraction and Comparison

This notebook extracts a continuous portion of a real steady 1D HEC-RAS reach into an independent project, validates the retained model content, runs both projects, and compares the retained-section results. The example uses the official **Balde Eagle Creek** project and keeps an intervening bridge structure.

The workflow demonstrates mechanics, not a universal acceptance standard. Choose station limits, boundary methods, and comparison tolerances using the governing project requirements and applicable HEC-RAS guidance.

In [ ]:
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

import ras_commander
from ras_commander import (
    RasBreakout1D,
    RasCmdr,
    RasExamples,
    RasPrj,
    init_ras_project,
)

print(f"ras-commander: {ras_commander.__version__}")
print(f"Loaded from: {ras_commander.__file__}")

## Configure the real example

The selected interval contains 11 natural cross sections and the bridge near river station 103245. The source plan is run first because an internal downstream cut needs source-plan water-surface results to create known-WSE boundary conditions for each steady profile.

HEC-RAS on Windows must see the working folder through a local or mapped-drive path; it cannot compute a project referenced through a UNC path.

In [ ]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "ras_commander").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
WORK_ROOT = REPO_ROOT / "working" / "235_1d_breakout_model"
RUN_ROOT = WORK_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
SOURCE_PLAN = "02"
RAS_VERSION = "7.0"
RIVER = "Bald Eagle"
REACH = "Loc Hav"
UPSTREAM_STATION = 106466.0
DOWNSTREAM_STATION = 98206.87

if str(RUN_ROOT).startswith("\\"):
    raise RuntimeError("Use a local or mapped-drive working path for HEC-RAS execution.")

RUN_ROOT.mkdir(parents=True, exist_ok=False)
print(f"Run workspace: {RUN_ROOT}")

## 1. Extract and initialize the source project

`RasExamples.extract_project()` creates a disposable working copy. The packaged source remains unchanged.

In [ ]:
source_path = RasExamples.extract_project(
    "Balde Eagle Creek",
    output_path=RUN_ROOT / "source",
)
source_ras = RasPrj()
init_ras_project(source_path, RAS_VERSION, ras_object=source_ras)

source_plan = source_ras.plan_df.loc[
    source_ras.plan_df["plan_number"].astype(str).str.zfill(2) == SOURCE_PLAN
].iloc[0]
display(
    source_ras.plan_df[[
        "plan_number", "Plan Title", "flow_type", "geometry_type",
        "Geom Path", "Flow Path",
    ]]
)

## 2. Run the source steady plan

The resulting plan HDF supplies profile-specific known water-surface elevations at the new downstream limit and provides the baseline results for comparison.

In [ ]:
source_compute = RasCmdr.compute_plan(
    SOURCE_PLAN,
    ras_object=source_ras,
    clear_geompre=True,
    force_rerun=True,
    num_cores=1,
    verify=True,
)
assert source_compute, "Source steady plan did not complete successfully"

source_plan_hdf = Path(f"{source_plan['full_path']}.hdf")
source_geometry = Path(source_plan["Geom Path"])
assert source_plan_hdf.is_file()
print(f"Source results: {source_plan_hdf}")

## 3. Resolve the retained reach slice

Selection is separate from writing. Other workflows can replace this call with `select_by_polygon()`, `select_by_network_segment()`, or `select_by_cross_sections()` and pass the resulting selection to the same extractor.

In [ ]:
selection = RasBreakout1D.select_by_stations(
    source_geometry,
    river=RIVER,
    reach=REACH,
    upstream_station=UPSTREAM_STATION,
    downstream_station=DOWNSTREAM_STATION,
)

display(pd.DataFrame({
    "river": [selection.river],
    "reach": [selection.reach],
    "upstream_station": [selection.upstream_station],
    "downstream_station": [selection.downstream_station],
    "retained_xs_count": [len(selection.stations)],
    "selector": [selection.selector],
}))
selection.stations

## 4. Write and structurally validate the breakout

The extractor copies complete retained cross-section and intervening structure blocks, carries the applicable flow changes, creates an independent `p01/g01/f01` project, zeroes the new downstream reach lengths, and assigns known-WSE downstream boundaries from the source results.

In [ ]:
breakout = RasBreakout1D.extract_selection(
    source_ras,
    RUN_ROOT / "breakout",
    selection,
    plan_number=SOURCE_PLAN,
    destination_name="Breakout",
    source_plan_hdf=source_plan_hdf,
    boundary_mode="auto",
)

display(breakout.validation.checks_df)
assert breakout.validation.is_valid
print(f"Boundary provenance: {breakout.boundary_provenance}")
print(f"Independent project: {breakout.project_file}")

## 5. Compare retained geometry before execution

This is an exact payload comparison for each retained cross section. The table attributes also report whether all intervening structure blocks match.

In [ ]:
geometry_comparison = RasBreakout1D.compare_geometry(
    source_geometry,
    breakout.geometry_file,
    breakout.selection,
)
display(geometry_comparison)
print(geometry_comparison.attrs)

assert geometry_comparison["content_equal"].all()
assert geometry_comparison.attrs["structure_blocks_equal"] is True

## 6. Run the independent breakout

Execution remains explicit. `RasBreakout1D.run()` delegates to `RasCmdr.compute_plan()` using the destination `RasPrj`.

In [ ]:
breakout_compute = RasBreakout1D.run(
    breakout,
    verify=True,
    force_rerun=True,
    num_cores=1,
)
assert breakout_compute, "Breakout plan did not complete successfully"

breakout_plan_hdf = Path(f"{breakout.plan_file}.hdf")
assert breakout_plan_hdf.is_file()
print(f"Breakout results: {breakout_plan_hdf}")

## 7. Compare retained-section hydraulic results

The comparison joins by river, reach, station, and profile and adds a delta for every numeric result available in both HDF files. All retained station/profile keys should join as `both`.

The new downstream cross section intentionally has zero reach lengths, so `channel_length_delta` is expected and is not a hydraulic mismatch. For this deterministic example, focus on flow, WSE, velocity, depth, top width, area, and slope deltas. Any project-specific acceptance threshold should be selected independently of this demonstration.

In [ ]:
results_comparison = RasBreakout1D.compare_results(
    source_plan_hdf,
    breakout_plan_hdf,
    breakout.selection,
)
assert results_comparison["_merge"].eq("both").all()

delta_columns = [
    column for column in results_comparison.columns
    if column.endswith("_delta") and column != "channel_length_delta"
]
delta_summary = (
    results_comparison[delta_columns]
    .abs()
    .agg(["count", "mean", "max"])
    .T
    .sort_index()
)
display(delta_summary)

preview_columns = [
    "river", "reach", "node_id", "profile",
    "flow_source", "flow_destination", "flow_delta",
    "wsel_source", "wsel_destination", "wsel_delta", "_merge",
]
display(results_comparison[preview_columns].head(16))

## Review products

The workflow leaves two independently openable HEC-RAS projects under `RUN_ROOT` and three audit products in memory:

- `breakout.validation.checks_df` — structural and relationship checks;
- `geometry_comparison` — exact retained geometry and structure-block agreement;
- `results_comparison` and `delta_summary` — retained-section/profile hydraulic differences.

For multi-reach, junction, lateral-structure, or unsteady extraction, stop at selection and use a workflow that explicitly supports those model elements; the current `RasBreakout1D` contract intentionally fails closed outside its one-reach steady scope.